# From Logits to Probabilities: Softmax, Cross-Entropy, and the Numerical Floor

**A research note on the map a language model actually optimizes.** Everything downstream — perplexity, temperature, top-k, top-p, the whole decoding stack — is a function of one object: the row of logits a model emits at a position. This study measures that map on real SmolLM2-135M logits: its invariances, its numerical failure boundary, and its exact identity with the training loss.

---

## Abstract

This artifact measures the logit → probability map on a real small language model, on CPU, fully reproducible. **What/how:** one forward pass over a fixed passage produces a `(T=83, V=49152)` logit tensor; from it we measure softmax shift invariance, the overflow boundary of the naive exponential against the log-sum-exp form, the three-way identity `CE = −log p_y = logsumexp(z) − z_y`, the temperature family `p_T = softmax(z/T)` with its entropy derivative `dH/dT = Var_{p_T}(z)/T³`, and the shape of real next-token distributions (`top1`, `k90`, effective support `exp(H)`). **What it shows:** the loss a model is trained on and the distribution a sampler draws from are the same object read two ways; the numerical form used to compute it is not cosmetic (the naive form dies at a locatable logit magnitude and is 140× less accurate even where it survives); and the width of a real distribution is set by its tail, not its head. **What it does not claim:** no calibration statement against ground-truth frequencies, no multi-model claim (that is the scale study), no decoding-quality judgement. Correctness rests on the numbers the cells produce, not on the framing.

## Related work, and what changes here

| Ref | Work | Contribution | Where this study goes further |
|---|---|---|---|
| Bridle (1990) | *Probabilistic interpretation of feedforward outputs* | Softmax as a normalized exponential with a temperature | We measure its invariances and its failure floor on a 49k-way real head |
| Bengio et al. (2003) | *A Neural Probabilistic Language Model*, JMLR | Fixes NLL of the next token as the LM objective | We verify `CE = logsumexp(z) − z_y` numerically, and locate where the two forms disagree |
| Goodfellow et al. (2016) | *Deep Learning*, §4.1 / §6.2 | The log-sum-exp trick as a stability requirement | We locate the exact logit magnitude where the naive form dies in float32 |
| Hinton et al. (2015) | *Distilling the Knowledge in a Neural Network*, arXiv:1503.02531 | Temperature-softened targets carry "dark knowledge" | We measure the entropy trajectory of that softening and verify its derivative identity |
| Guo et al. (2017) | *On Calibration of Modern Neural Networks*, arXiv:1706.04599 | Temperature scaling as post-hoc calibration | We treat `T` as a distribution-shape operator and quantify shape, not calibration |
| Holtzman et al. (2020) | *The Curious Case of Neural Text Degeneration*, arXiv:1904.09751 | Real LM tails are heavy; truncation is necessary | We supply the measurement that motivates it: `k90` vs `exp(H)` per position |
| Allal et al. (2025) | *SmolLM2*, arXiv:2502.02737 | The model family under measurement | A 49,152-way head on a 576-wide model — an 85× vocab/width ratio |

**The differentiator.** The softmax identity is textbook; what is rarely *measured* is where the textbook form breaks in float32, how much accuracy the stable form buys where both survive, and how far a real 49k-way head sits from uniform at a given position. All three are numbers produced here.

## The measurement object and metrics

**Object.** One forward pass of SmolLM2-135M over a fixed passage yields logits `Z ∈ R^{T×V}`, `V = 49,152`. Every quantity below is a function of one row `z = Z[t] ∈ R^V`. The row is `h_t · W_Eᵀ` — the final hidden state read against all 49,152 embedding rows. Unnormalized, unbounded, and predicting token `t+1`, not token `t`.

**The map.**

$$p_i = \frac{e^{z_i}}{\sum_j e^{z_j}}, \qquad \log p_i = z_i - \operatorname{logsumexp}(z), \qquad \operatorname{logsumexp}(z) = m + \log\sum_j e^{z_j - m},\quad m = \max_j z_j$$

Two structural facts, tested rather than assumed:

- **Shift invariance.** `softmax(z + c·1) = softmax(z)`, because `e^c` cancels between numerator and denominator. Softmax is a function of logit *differences* only: `z` carries one redundant degree of freedom, and the map lands on a `V−1` dimensional simplex. This is *why* subtracting `m` is free, and the only reason a 49k-way exponential is computable in finite precision.
- **Scale sensitivity.** `softmax(a·z) ≠ softmax(z)` for `a ≠ 1`. Invariant to translation, not to dilation — and temperature is exactly that dilation.

**The loss.** For true next token `y`:

$$\mathcal{L}(z, y) = -\log p_y = -\big(z_y - \operatorname{logsumexp}(z)\big) = \operatorname{logsumexp}(z) - z_y, \qquad \frac{\partial \mathcal{L}}{\partial z_i} = p_i - \mathbb{1}[i=y]$$

No margin, no regularizer, no second term. Each token's logit is pushed down in proportion to the probability the model currently assigns it; only the true token is pushed up. The tokens compete for one budget of mass — a language model is a density model, not 49,152 independent classifiers.

**Shape metrics.** For the distribution `p` at a position:
- `top1 = max_i p_i` — the leading token's mass.
- `H(p) = −Σ_i p_i log p_i` in nats — the mean surprise the distribution assigns to its own draws; `0 ≤ H ≤ log V = 10.803`.
- `exp(H)` — the **effective support**: the size of the uniform distribution with the same entropy, readable directly as a token count.
- `k90` — the smallest `m` whose `m` largest probabilities sum to ≥ 0.90.

`exp(H)` and `k90` are *not* the same statistic. `exp(H)` is mass-weighted and dominated by the head; `k90` is a raw count and must walk the tail. Where they disagree, the tail is doing the work — and that gap is the object every truncation operator in the later studies cuts.

## Hypothesis board (decided before measurement)

| # | Claim (falsifiable) | Predicted |
|---|---|---|
| H1 | `softmax(z + c)` equals `softmax(z)` to float noise for moderate `c` | max abs diff ≤ 1e-7 |
| H2 | The naive `exp(z)/Σ exp(z)` breaks before the stable form does | naive → `nan` once the shifted max crosses `log(3.4e38) ≈ 88.72` |
| H3 | `−log p_y`, `logsumexp(z) − z_y`, and `F.cross_entropy` are the same number | agreement ≤ 1e-6 |
| H4 | Entropy is strictly increasing in `T`, with `H → log V = 10.803` as `T → ∞` | monotone; `H(T=100)` within 0.05 of the ceiling |
| H5 | Real next-token distributions are far from uniform | median `k90` < 50, median `exp(H)` < 200 (against `V = 49,152`) |

**How a verdict is won.** H1–H3 are exact-identity claims: they hold only if the residual sits at or below the float32 epsilon scale (`≈1.19e-7`). H4 is a monotonicity claim over the whole sweep — a single inversion falsifies it. H5 is distributional, read off the median over all measured positions, never a chosen row.

**Pre-committed honesty note.** One model, one passage, final-layer logits only, float32 throughout. Positions are not i.i.d. — they come from one document, so the medians describe *this* text under *this* model, not a corpus. The identities, by contrast, are properties of the map and hold for any logit vector.

## Protocol & constraints

- **Model:** `HuggingFaceTB/SmolLM2-135M`, `dtype=torch.float32`, `eval()`, inside `torch.no_grad()`. Float32 is deliberate: the numerical claims are about float32 boundaries.
- **Offline reuse:** `HF_HUB_OFFLINE=1`; weights resolve from the shared local hub cache. No model file is copied into this repository.
- **Text:** one fixed passage, `add_special_tokens=False`, giving 83 positions.
- **Cache:** the logit tensor and token ids are written once to `pps_cache/logits_135M.pt`; every later study reads that cache rather than re-loading a model into the kernel.
- **Determinism:** nothing is sampled here — every reported quantity is a deterministic function of `Z`.

**Naming convention used throughout the repository.** Capital `Z`, `P` are batched `(T, V)`; lowercase `z`, `p` are one position. A subscript that indexes is a coordinate (`z_y`, `p_i`); a subscript that names a knob is a family member (`p_T`, `p_β`). `p_1` is reserved for the untouched model distribution at `T = 1`, since every operator in the later studies is defined as a map `p_1 → p'`. Note the one overload: `T, V = Z.shape` makes `T` the token count, so the temperature loop variable is `Tt`.

## The logit rig

**Why:** every measurement in this note is a function of one `(T, V)` tensor. Producing it once, caching it, and reporting its gross statistics is the plumbing the rest of the study stands on.

**The method:** tokenize a fixed passage without special tokens; if the cache blob exists load `{logits, ids}` from it, otherwise load the model in float32, run one `no_grad` forward, keep `logits[0]`, save, then `del` the model and `gc.collect()`. Report shape, logit min/max/mean/std, the median per-row spread `max(z) − min(z)`, `log V`, and the headroom to the float32 `exp` wall.

**What the run shows:** `V = 49,152` on a 576-wide model — an 85× vocab-to-width ratio. Max logit 33.881 leaves **54.84 of headroom** before a naive exponential overflows: the model's own raw scale already consumes 38% of the float32 exponent budget. The median spread of **34.694 nats** means the top token of a typical row is `e^34.69 ≈ 1.2×10^15` times likelier than the bottom one — the row is violently peaked before softmax runs, which is the mechanism behind the `k90 ≪ V` result at the end of this study.

In [1]:
import os
os.environ["HF_HUB_OFFLINE"] = "1"          # reuse the local HF cache, no network

import gc, torch, torch.nn.functional as F
from pathlib import Path
from transformers import AutoTokenizer, AutoModelForCausalLM

REPO  = "HuggingFaceTB/SmolLM2-135M"
CACHE = Path("pps_cache"); CACHE.mkdir(exist_ok=True)
BLOB  = CACHE / "logits_135M.pt"

TEXT = (
    "The transformer architecture replaced recurrence with attention, and in doing so "
    "made the cost of a sequence quadratic in its length but perfectly parallel across "
    "positions. A language model trained this way does not store sentences; it stores a "
    "conditional distribution over the next token given every prefix it has seen. "
    "Sampling from that distribution is the only thing generation ever does, and every "
    "decoding heuristic is a modification of the distribution rather than of the model."
)

tok = AutoTokenizer.from_pretrained(REPO)
ids = tok(TEXT, return_tensors="pt", add_special_tokens=False)["input_ids"]

if BLOB.exists():
    blob = torch.load(BLOB, map_location="cpu", weights_only=True)
    Z, ids = blob["logits"], blob["ids"]
else:
    model = AutoModelForCausalLM.from_pretrained(REPO, dtype=torch.float32).eval()
    with torch.no_grad():
        Z = model(ids).logits[0].float()          # (T, V)
    torch.save({"logits": Z, "ids": ids}, BLOB)
    del model; gc.collect()

T, V = Z.shape
logV = torch.log(torch.tensor(float(V)))
spread = (Z.max(-1).values - Z.min(-1).values).median()

print(f"tokens T = {T}   vocab V = {V}   dtype {Z.dtype}")
print(f"logit  min {Z.min():.3f}  max {Z.max():.3f}  mean {Z.mean():.3f}  std {Z.std():.3f}")
print(f"median per-row spread (max-min): {spread:.3f}")
print(f"log V = {logV:.4f} nats   (uniform-entropy ceiling)")
print(f"float32 exp limit: log(3.40e38) = 88.72  -> headroom = {88.72 - Z.max():.2f}")

tokens T = 83   vocab V = 49152   dtype torch.float32
logit  min -27.402  max 33.881  mean 7.771  std 5.146
median per-row spread (max-min): 34.694
log V = 10.8027 nats   (uniform-entropy ceiling)
float32 exp limit: log(3.40e38) = 88.72  -> headroom = 54.84

## Shift invariance and the point where the naive form dies

**Why:** the invariance is the entire justification for the max-subtraction inside every framework implementation. It is exact in real arithmetic; in float32 it is exact only until `exp` overflows. Locating that boundary turns folklore into a number.

**The derivation.** Adding `c` to every coordinate multiplies numerator and denominator by the same `e^c`:

$$\text{softmax}(z+c)_i=\frac{e^{c}e^{z_i}}{e^{c}\sum_j e^{z_j}}=\text{softmax}(z)_i, \qquad \operatorname{logsumexp}(z+c) = \operatorname{logsumexp}(z) + c$$

**Why it is not free in float32.** Real arithmetic cancels `e^c` perfectly; floating point must *materialize* `e^{z_i+c}` first. IEEE-754 float32 tops out at `3.40e38`, so `e^u = inf` for `u > 88.72`. One `inf` in both numerator and denominator gives `inf/inf = nan` — a single coordinate destroys all 49,152 entries. The failure is total, not gradual.

**What the stable form does.** With `c = −m`, every exponent argument is `≤ 0`, every term lies in `(0, 1]`, and the largest is exactly `1`. Overflow becomes *structurally impossible*. The only error left is underflow of terms with `z_i − m < −87.3`, which is benign: those tokens carry probability below `1e-38`. The asymmetry is the point — overflow is catastrophic, underflow is harmless, so the trick trades one for the other.

**The method:** take `z = Z[10]` and reference `p_ref = softmax(z)`; compare the stable path and a deliberately naive `exp(v)/exp(v).sum()` across `c ∈ {−100,−10,0,10,100}` by max absolute deviation and finiteness; scan `c` upward for the first non-finite result; verify the log-space shift identity; count underflowed terms.

**What the run shows.** The wall was *predicted, not discovered*: row max 29.135 means the naive form must die at `c = 88.72 − 29.135 = 59.585`. The scan stepped by 5 and brackets it exactly — `c = 55` finite, `c = 60` gives `nan` at shifted max 89.13.

The stable column is not uniformly zero, and the pattern is the real content:

| c | stable residual | why |
|---|---|---|
| 0 | 0.000e+00 | identical computation, bit-for-bit |
| −10 | 2.255e-17 | `z+c` stays in the same binade → the addition is *exact* |
| +10 | 5.960e-08 | crosses into a larger binade, ulp doubles → one rounding |
| ±100 | 1.192e-07 | exponent change costs the last mantissa bit → exactly float32 eps |

That residual is not softmax error at all — it is the error of *representing* `z + c` before softmax ever runs. The invariance holds to the limit of the number system and no worse.

The naive form carries a floor of **1.681e-05 even where it is finite** — constant across shifts, ~141× eps, from exponentiating and summing 49,152 terms spanning 24 orders of magnitude. It sits between the pairwise-summation bound (`log₂V · eps ≈ 1.9e-6`) and the random-walk estimate (`√V · eps ≈ 2.6e-5`). So the max-subtraction is not only an anti-overflow device: it buys **two orders of magnitude of accuracy** in the regime where both forms work. At `c = −100` the floor rises to 1.705e-05 as the exponentials edge toward subnormals.

Underflow count: **0 of 49,152**. This row's spread is under 87.3 nats, so the stable form is lossless in both directions here — it is not trading overflow for underflow, it gets the trade for free.

In [2]:
z = Z[10].clone()                        # one real logit row
p_ref = torch.softmax(z, dim=-1)

def naive_softmax(v):
    e = torch.exp(v)                     # no max subtraction
    return e / e.sum()

print(f"row max logit = {z.max():.3f}   float32 exp overflow at log(3.40e38) = 88.72\n")
print(f"{'shift c':>9} | {'stable max|dp|':>15} | {'naive max|dp|':>14} | naive finite")
for c in [-100.0, -10.0, 0.0, 10.0, 100.0]:
    zs = z + c
    p_stable = torch.softmax(zs, dim=-1)
    p_naive  = naive_softmax(zs)
    d_stable = (p_stable - p_ref).abs().max().item()
    d_naive  = (p_naive  - p_ref).abs().max().item()
    print(f"{c:9.1f} | {d_stable:15.3e} | {d_naive:14.3e} | {bool(torch.isfinite(p_naive).all())}")

first_fail = next(c for c in range(0, 200, 5)
                  if not torch.isfinite(naive_softmax(z + c)).all())
print(f"\nnaive form first non-finite at shift c = {first_fail}  "
      f"(shifted max logit = {z.max().item() + first_fail:.2f}, wall = 88.72)")

lse = torch.logsumexp(z, -1).item()
print(f"logsumexp(z+100) = {torch.logsumexp(z + 100, -1).item():.4f}  vs  "
      f"logsumexp(z)+100 = {lse + 100:.4f}   (exact shift identity)")
print(f"underflow check: {(z - z.max() < -87.3).sum().item()} of {z.numel()} "
      f"terms flush to zero in the stable form")

row max logit = 29.135   float32 exp overflow at log(3.40e38) = 88.72

  shift c |  stable max|dp| |  naive max|dp| | naive finite
   -100.0 |       1.192e-07 |      1.705e-05 | True
    -10.0 |       2.255e-17 |      1.681e-05 | True
      0.0 |       0.000e+00 |      1.681e-05 | True
     10.0 |       5.960e-08 |      1.681e-05 | True
    100.0 |       1.192e-07 |            nan | False

naive form first non-finite at shift c = 60  (shifted max logit = 89.13, wall = 88.72)
logsumexp(z+100) = 129.1852  vs  logsumexp(z)+100 = 129.1852   (exact shift identity)
underflow check: 0 of 49152 terms flush to zero in the stable form

## Cross-entropy is one entry of the distribution

**Why:** the loss is not a construction bolted onto the distribution — it is a lookup into it. Writing the identity three ways and measuring the residuals removes any ambiguity about what the model was optimized for, and fixes the exact quantity perplexity averages in the next study.

**The causal shift.** Row `t` predicts token `t+1`, so `Z[:-1]` is scored against `y = ids[0, 1:]`, giving 82 scored positions from 83 tokens. Off-by-one here is the most common perplexity bug in the wild.

**Why three routes, and why they should differ.** Algebraically identical, numerically not:

- **gather from `log_softmax`** — normalizes in log space with the max-subtraction built in, then indexes. Most accurate.
- **`logsumexp(z) − z_y`** — subtracts two numbers of magnitude ~30 to yield ~4. Catastrophic cancellation: the leading digits annihilate and the result inherits the *absolute* error of the operands, not their relative error.
- **`F.cross_entropy`** — the fused kernel; the same algorithm as the first, so it should agree bit-for-bit.

**Units.** NLL is in nats. `L = 0` is certainty; `L = log V = 10.803` is exactly as surprised as a uniform guess; `L > 10.803` means the model actively bet against the truth.

**What the run shows.** `gather ≡ F.cross_entropy` at **exactly 0.0** — same kernel, confirmed. The closed form is off by **2.97e-05**, ~250× eps: cancellation, as predicted. Relative to the ~34-magnitude operands that is ~7 eps, textbook behavior, but it falsifies the ≤1e-6 prediction — the identity is exact in algebra and loses about 2.4 decimal digits in float32.

Mean NLL **4.343** against median **3.610** — right-skewed. This matters because perplexity is `exp` of the arithmetic mean of logs, i.e. the **geometric mean of `1/p_y`**: rare catastrophes multiply into it. Four positions of 82 beat `log V`, and those four alone pull the passage perplexity of **76.94** well above what the typical token deserves.

The two extremes are different problems, not different difficulties. `' than'` after `"...rather"` at `p = 0.995` is syntax — a bigram constraint. `' recurrence'` after `"The transformer architecture replaced"` at `p = 1.79e-08` is *knowledge*; the model answered `' the'` (0.484), a grammatically flawless continuation. The loss is not measuring fluency, it is measuring content the 135M parameters failed to store — the first concrete reason perplexity is a weak proxy for quality.

In [3]:
Zc = Z[:-1]                       # row t predicts token t+1
y  = ids[0, 1:]                   # targets
n  = Zc.shape[0]
idx = torch.arange(n)

logp   = torch.log_softmax(Zc, dim=-1)
loss_a = -logp[idx, y]                                        # gather from log_softmax
loss_b = torch.logsumexp(Zc, dim=-1) - Zc[idx, y]             # closed form (cancellation)
loss_c = F.cross_entropy(Zc, y, reduction="none")             # fused framework kernel

print(f"positions scored: {n}")
print(f"max |a - b| = {(loss_a - loss_b).abs().max():.3e}   <- cancellation in the closed form")
print(f"max |a - c| = {(loss_a - loss_c).abs().max():.3e}")
print(f"max |b - c| = {(loss_b - loss_c).abs().max():.3e}")
print(f"float32 eps = {torch.finfo(torch.float32).eps:.3e}\n")

print(f"mean NLL  = {loss_a.mean():.4f} nats/token")
print(f"exp(mean) = {loss_a.mean().exp():.3f}   <- perplexity of this passage")
print(f"per-token NLL: min {loss_a.min():.4f}  median {loss_a.median():.4f}  max {loss_a.max():.4f}")
print(f"positions worse than uniform (NLL > log V = {logV:.3f}): "
      f"{(loss_a > logV).sum().item()} of {n}")

hard, easy = int(loss_a.argmax()), int(loss_a.argmin())
for tag, i in [("hardest", hard), ("easiest", easy)]:
    ctx = tok.decode(ids[0, max(0, i-6):i+1])
    print(f"\n{tag} position {i}: after {ctx!r}")
    print(f"   true token {tok.decode(y[i:i+1])!r}  NLL {loss_a[i]:.3f}  p_y {torch.exp(-loss_a[i]):.2e}")
    top = torch.topk(torch.softmax(Zc[i], -1), 3)
    print(f"   model's top-3: {[(tok.decode([int(j)]), round(float(v),3)) for v, j in zip(top.values, top.indices)]}")

positions scored: 82
max |a - b| = 2.970e-05   <- cancellation in the closed form
max |a - c| = 0.000e+00
max |b - c| = 2.970e-05
float32 eps = 1.192e-07

mean NLL  = 4.3430 nats/token
exp(mean) = 76.941   <- perplexity of this passage
per-token NLL: min 0.0051  median 3.6098  max 17.8405
positions worse than uniform (NLL > log V = 10.803): 4 of 82

hardest position 3: after 'The transformer architecture replaced'
   true token ' recurrence'  NLL 17.841  p_y 1.79e-08
   model's top-3: [(' the', 0.484), (' by', 0.097), (' a', 0.062)]

easiest position 77: after ' is a modification of the distribution rather'
   true token ' than'  NLL 0.005  p_y 9.95e-01
   model's top-3: [(' than', 0.995), (' that', 0.001), (' then', 0.001)]

## Temperature as a power transform on the distribution

**Why:** temperature is the only decoding knob that reshapes the *whole* distribution instead of truncating it. Everything else in this repository operates on the curve temperature produces.

**Two equivalent readings.** Dividing logits is a power law on probabilities:

$$p_{T,i} \;\propto\; e^{z_i/T} = \left(e^{z_i}\right)^{1/T} \;\propto\; \left(p_{1,i}\right)^{1/T}, \qquad \log\frac{p_{T,i}}{p_{T,j}} = \frac{z_i - z_j}{T}$$

`T < 1` raises the base distribution to a power > 1 → large probabilities shrink slower than small ones → sharpening. `T > 1` lifts the tail toward the head → flattening. All pairwise log-odds scale linearly, so temperature **never reorders tokens**: the argmax is `T`-invariant. It only rescales how much the ranking matters.

**Why entropy must increase, exactly.** With `β = 1/T`, `log Z(β) = log Σ_j e^{βz_j}` is the cumulant generating function of `z` under `p_β`: `d log Z/dβ = E_β[z]` and `d²log Z/dβ² = Var_β(z)`. Since `H = log Z − β E_β[z]`,

$$\frac{dH}{d\beta} = -\beta\operatorname{Var}_\beta(z), \qquad \frac{dH}{dT} = \frac{dH}{d\beta}\cdot\frac{d\beta}{dT} = \frac{\operatorname{Var}_{p_T}(z)}{T^{3}} \;\ge\; 0$$

A variance cannot be negative, so entropy is monotone in `T` for *any* logit vector — no exceptions, no model dependence. Note `Var_{p_T}(z)` is the variance of the **logit values under the current distribution**, not the variance of `p`. The identity also predicts *where* the motion is: `dH/dT` vanishes at both ends, where the distribution has collapsed onto one token (`Var → 0`) or where `T³` overwhelms a frozen variance.

**The limits.** `T→0⁺`: `p →` one-hot(argmax `z`), `H → 0`. `T→∞`: all logits equal, `p →` Uniform(V), `H → log V = 10.803`.

**Ordering rule.** `T` acts on logits, *before* any truncation. Top-k/top-p then cut the reshaped curve. Reversing the order gives a different distribution — the deployment study measures that.

**The method:** sweep `T ∈ {0.1 … 100}` on one real row reporting `top1`, `H`, `exp(H)`, `k90`; test monotonicity and the ceiling gap; check argmax invariance; then compare a central finite difference of `H(T)` against `Var_{p_T}(z)/T³`.

**What the run shows.** `T ≤ 1` is dead flat — `top1` 0.95, `k90` 1. With a 35-nat spread there is nothing left to sharpen. Between `T = 1` and `T = 2`, `exp(H)` goes 1.3 → 2302.8: a **1,700× explosion in effective support over one unit of T**, with `k90` running 1 → 535 → 15,620. That is the entire practical range of the knob, and it says `T = 1.5` is not "slightly more creative" — it is a different distribution.

The derivative check closes the derivation: 0.12590/0.12695, 1.17165/1.15723, 6.91462/6.91421, 0.08202/0.08464 — agreement at the `1e-3` finite-difference truncation error. The non-monotone `Var/T³` column is the real content: entropy motion **peaks near `T = 2`** and dies at both ends. The same `ΔT` buys 55× more entropy at the peak than at `T = 0.5`, which is exactly why the knob feels gentle below 1 and cliff-like above 1.5.

In [4]:
import numpy as np

def shape_stats(p):
    H   = -(p * torch.log(p.clamp_min(1e-30))).sum()
    srt = torch.sort(p, descending=True).values
    k90 = int((torch.cumsum(srt, 0) < 0.90).sum().item()) + 1
    return float(srt[0]), float(H), float(H.exp()), k90

z  = Z[10]
Ts = [0.1, 0.25, 0.5, 0.7, 1.0, 1.5, 2.0, 5.0, 100.0]

print(f"{'T':>7} | {'top1':>8} | {'H (nats)':>9} | {'exp(H)':>10} | {'k90':>6}")
Hs = []
for Tt in Ts:
    p = torch.softmax(z / Tt, dim=-1)
    t1, H, eH, k90 = shape_stats(p)
    Hs.append(H)
    print(f"{Tt:7.2f} | {t1:8.4f} | {H:9.4f} | {eH:10.1f} | {k90:6d}")

print(f"\nlog V = {float(logV):.4f} nats   (uniform ceiling, exp = {V})")
print(f"H strictly increasing in T: {all(b > a for a, b in zip(Hs, Hs[1:]))}")
print(f"|H(T=100) - log V| = {abs(Hs[-1] - float(logV)):.4f}")

p0 = torch.softmax(z / 0.01, dim=-1)
print(f"T=0.01 -> top1 {p0.max():.6f}, token {tok.decode([int(p0.argmax())])!r}  (one-hot limit)")
print(f"argmax invariant across T: "
      f"{len({int(torch.softmax(z/t, -1).argmax()) for t in Ts}) == 1}")

# does dH/dT = Var_{p_T}(z) / T^3 hold numerically?
def entropy_at(t):
    p = torch.softmax(z / t, dim=-1)
    return float(-(p * torch.log(p.clamp_min(1e-30))).sum())

print(f"\n{'T':>6} | {'dH/dT (finite diff)':>20} | {'Var(z)/T^3':>12}")
for Tt in [0.5, 1.0, 2.0, 5.0]:
    h  = 1e-3
    fd = (entropy_at(Tt + h) - entropy_at(Tt - h)) / (2 * h)
    p  = torch.softmax(z / Tt, dim=-1)
    var = float((p * z**2).sum() - (p * z).sum()**2)
    print(f"{Tt:6.2f} | {fd:20.5f} | {var / Tt**3:12.5f}")

      T |     top1 |  H (nats) |     exp(H) |    k90
   0.10 |   1.0000 |    0.0000 |        1.0 |      1
   0.25 |   1.0000 |    0.0000 |        1.0 |      1
   0.50 |   0.9987 |    0.0102 |        1.0 |      1
   0.70 |   0.9903 |    0.0600 |        1.1 |      1
   1.00 |   0.9508 |    0.2682 |        1.3 |      1
   1.50 |   0.6847 |    2.5674 |       13.0 |    535
   2.00 |   0.1989 |    7.7419 |     2302.8 |  15620
   5.00 |   0.0012 |   10.6032 |    40264.6 |  36768
 100.00 |   0.0000 |   10.8022 |    49128.7 |  43966

log V = 10.8027 nats   (uniform ceiling, exp = 49152)
H strictly increasing in T: True
|H(T=100) - log V| = 0.0005
T=0.01 -> top1 1.000000, token ' so'  (one-hot limit)
argmax invariant across T: True

     T |  dH/dT (finite diff) |   Var(z)/T^3
  0.50 |              0.12590 |      0.12695
  1.00 |              1.17165 |      1.15723
  2.00 |              6.91462 |      6.91421
  5.00 |              0.08202 |      0.08464

## The shape of real next-token distributions

**Why:** every truncation operator in the later studies is a bet about how much of the 49,152-way vector is actually live. That bet needs a measured baseline: how wide is a real distribution, and how much does the width vary across positions of one document?

**The method:** softmax all rows at `T = 1`; compute per-row entropy, sort descending, cumsum to `k90`; report median/min/max of `top1`, `H`, `exp(H)`, `k90`, the ratios against `V`, and the tail-weight ratio `k90 / exp(H)`; identify the widest and tightest positions; plot the sorted mass of both on log-log axes against a `1/V` reference, and the histogram of `k90`.

**What the run shows.** Median `top1` 0.259, median `exp(H)` **66.6**, median `k90` **222** — 0.45% of the vocabulary. Real distributions live in a tiny corner of the simplex. But the prediction of `k90 < 50` fails by 4.4×, and the reason is the 3.3× gap between the two width statistics: the typical position has ~67 tokens' worth of *mass* smeared over ~222 tokens of *support*. The prediction estimated the head; the tail did the work.

The extremes bracket the entire argument for adaptive truncation. Position 0 has no context at all — `H` 8.647 of a possible 10.803, `k90` **12,097**, `top1` 0.022; with nothing to condition on the model is near-uniform over a quarter of the vocabulary. Position 77 (`"...rather"` → `" than"`) has `H` 0.053 and `k90` **1**. One position needs 12,097 tokens, another needs 1, inside the same 83-token document. No single fixed `k` serves both.

**The left panel (sorted mass, log-log).** The tight curve starts at 0.995 and falls off a cliff — nothing past rank 1 carries mass. The wide curve starts near 0.02 and descends as a near-straight line, which on log-log axes means a **power law**, `p_rank ∝ rank^{−α}`. A power-law tail has no natural cutoff; it simply continues, thousands of tokens each carrying a little, collectively carrying a lot. The gray `1/V` line marks uniform: the wide curve crosses it around rank 12k, so tokens beyond that are *less* likely than a blind draw over the whole vocabulary. Everything between rank ~100 and that crossing is the unreliable tail — individually negligible, collectively substantial, and the direct source of degeneration when sampled from.

**The right panel (histogram of `k90`).** Heavily right-skewed with a spike at small `k90` and a long thin run out to 12,097. Most positions are syntactically pinned and need a handful of tokens; a few are wide open. The mean would sit far above the median 222, which is why the median is the honest summary — and the shape of this histogram *is* the empirical argument for mass-based truncation over count-based truncation.

In [5]:
import matplotlib.pyplot as plt

P     = torch.softmax(Z, dim=-1)
H_all = -(P * torch.log(P.clamp_min(1e-30))).sum(-1)
srt   = torch.sort(P, dim=-1, descending=True).values
k90_all  = (torch.cumsum(srt, -1) < 0.90).sum(-1) + 1
top1_all = srt[:, 0]

print(f"across {T} positions (T = 1.0):")
for nm, v in [("top1", top1_all), ("H (nats)", H_all),
              ("exp(H)", H_all.exp()), ("k90", k90_all.float())]:
    print(f"  {nm:9s} median {v.median():10.3f} | min {v.min():9.3f} | max {v.max():10.3f}")
print(f"\nV = {V};  median exp(H)/V = {float(H_all.exp().median())/V:.5f};"
      f"  median k90/V = {float(k90_all.float().median())/V:.5f}")
print(f"median k90 / median exp(H) = {float(k90_all.float().median())/float(H_all.exp().median()):.1f}x"
      f"   <- tail weight: how far the count exceeds the mass-weighted width")

hi, lo = int(H_all.argmax()), int(H_all.argmin())
print(f"widest   position {hi}: H {H_all[hi]:.3f}, k90 {int(k90_all[hi])}, top1 {top1_all[hi]:.4f}")
print(f"tightest position {lo}: H {H_all[lo]:.3f}, k90 {int(k90_all[lo])}, top1 {top1_all[lo]:.4f}")

fig, ax = plt.subplots(1, 2, figsize=(12, 4))
ax[0].loglog(np.arange(1, 2001), srt[hi, :2000].numpy(), label=f"widest  (H={H_all[hi]:.2f})")
ax[0].loglog(np.arange(1, 2001), srt[lo, :2000].numpy(), label=f"tightest (H={H_all[lo]:.2f})")
ax[0].axhline(1.0 / V, ls="--", c="gray", lw=1, label="uniform 1/V")
ax[0].set_xlabel("rank"); ax[0].set_ylabel("probability")
ax[0].set_title("sorted probability mass, two real positions")
ax[0].legend(fontsize=8); ax[0].grid(alpha=0.3)

ax[1].hist(k90_all.numpy(), bins=40, color="tab:blue", alpha=0.8)
ax[1].set_xlabel("k90 (tokens covering 90% mass)"); ax[1].set_ylabel("positions")
ax[1].set_title(f"k90 across {T} positions (V = {V})"); ax[1].grid(alpha=0.3)
plt.tight_layout(); plt.show(); plt.close("all")

across 83 positions (T = 1.0):
  top1      median      0.259 | min     0.022 | max      0.995
  H (nats)  median      4.198 | min     0.053 | max      8.647
  exp(H)    median     66.574 | min     1.054 | max   5691.697
  k90       median    222.000 | min     1.000 | max  12097.000

V = 49152;  median exp(H)/V = 0.00135;  median k90/V = 0.00452
median k90 / median exp(H) = 3.3x   <- tail weight: how far the count exceeds the mass-weighted width
widest   position 0: H 8.647, k90 12097, top1 0.0219
tightest position 77: H 0.053, k90 1, top1 0.9949

## Findings

| # | Claim | Predicted | Measured | Verdict |
|---|---|---|---|---|
| H1 | shift invariance holds to float noise | ≤ 1e-7 | **1.192e-07** = float32 eps, worst case over `c ∈ [−100, 100]` | ✅ holds |
| H2 | naive softmax overflows, stable form does not | `nan` once shifted max > 88.72 | `nan` at `c = 60` → shifted max **89.13**; analytic crossing 59.585 bracketed by the step-5 scan | ✅ holds |
| H3 | three routes to cross-entropy agree | ≤ 1e-6 | gather ≡ `F.cross_entropy` at **0.0**; closed form off by **2.97e-05** (≈250× eps) | ⚠️ partial — cancellation, not disagreement |
| H4 | entropy strictly increasing in `T`, ceiling `log V` | monotone, within 0.05 | monotone **True**; gap **0.0005**; `dH/dT` matches `Var/T³` to 1e-3; argmax `T`-invariant | ✅ holds |
| H5 | real distributions are far from uniform | median `k90` < 50, median `exp(H)` < 200 | median `exp(H)` **66.6** ✓; median `k90` **222** ✗ (4.4× over) | ⚠️ partial — the tail is heavier than predicted |

**Secondary measurements worth carrying forward.** The naive softmax floor of **1.681e-05** where it is still finite (141× eps) — the max-subtraction buys two orders of accuracy, not just safety. The **1,700× jump in `exp(H)` between `T = 1` and `T = 2`**, with `dH/dT` peaking near `T = 2`. And the **1 → 12,097 range of `k90` inside one document**.

## Discussion and verdict

**The claim in one line.** The training loss and the sampling distribution are the same 49,152-way object read two ways, and its practically relevant width is set by its **tail, not its head** — median `exp(H)` 66.6 against median `k90` 222, ranging from 1 to 12,097 across a single document.

**The three reversals, and what each teaches.**

*H3 — the closed form is not the safe form.* `logsumexp(z) − z_y` is the identity everyone writes on the board, and it is 250× less accurate than gathering from `log_softmax` because it subtracts two ~34-magnitude numbers to produce a ~4-magnitude answer. Algebraic identity does not imply numerical interchangeability; the gather route is the one to ship.

*H5 — the head is not the distribution.* Predicting `k90 < 50` from a median `top1` of 0.259 was reasonable and wrong. The 3.3× gap between `k90` and `exp(H)` is the tail asserting itself: ~67 tokens' worth of mass smeared across ~222 tokens of support. Any operator tuned on head statistics will mis-size its cut.

*The fixed-`k` impossibility.* One document contains a position needing 12,097 tokens to reach 90% mass and a position needing 1. This is measured, not argued, and it is the structural case for mass-based truncation — the subject of the synthetic-control study two steps ahead.

**Why the numerical section is not a detour.** Two of the five hypotheses were about floating point, and both produced findings that change practice: the overflow wall is *predictable* from `max(z)` (headroom 54.84 here), and the stable form is 141× more accurate even far from that wall. Every perplexity number in this repository is an exponential of an average of logs; errors at the 1e-5 level are invisible in one token and compound across a corpus.

**Honesty note.** One model, one passage, final-layer logits, float32. The distributional numbers (`k90`, `exp(H)`, the medians) describe this text under this model. The identities — shift invariance, the cross-entropy equality, `dH/dT = Var/T³`, the entropy limits — are properties of the map and hold for any logit vector, which is why they were the claims worth pre-committing.

**Where the next study goes.** Cross-entropy is measured per position here; the next study averages it into perplexity — `PPL = exp(mean NLL)`, the geometric mean of `1/p_y`, measured at **76.94** on this passage. Averaging raises the question this study cannot answer: what does that number depend on besides the model? Context length, stride, and tokenizer all enter, and each one moves it.

## References

1. Bridle, J. S. (1990). *Probabilistic Interpretation of Feedforward Classification Network Outputs*. Neurocomputing, NATO ASI Series.
2. Bengio, Y., Ducharme, R., Vincent, P., Jauvin, C. (2003). *A Neural Probabilistic Language Model*. JMLR 3:1137–1155.
3. Goodfellow, I., Bengio, Y., Courville, A. (2016). *Deep Learning*, MIT Press — §4.1 (numerical stability), §6.2 (softmax output units).
4. Hinton, G., Vinyals, O., Dean, J. (2015). *Distilling the Knowledge in a Neural Network*. arXiv:1503.02531.
5. Guo, C., Pleiss, G., Sun, Y., Weinberger, K. (2017). *On Calibration of Modern Neural Networks*. arXiv:1706.04599.
6. Holtzman, A., Buys, J., Du, L., Forbes, M., Choi, Y. (2020). *The Curious Case of Neural Text Degeneration*. arXiv:1904.09751.
7. Allal, L. B., Lozhkov, A., et al. (2025). *SmolLM2: When Smol Goes Big*. arXiv:2502.02737.
8. Higham, N. J. (2002). *Accuracy and Stability of Numerical Algorithms*, 2nd ed., SIAM — cancellation and summation error bounds.